In [1]:
import pandas as pd
from dataset_class.job_post_dataset import JobPostingDataset
from sklearn.model_selection import train_test_split
import nltk
from nltk.tokenize import word_tokenize
from collections import Counter
import numpy as np
from gensim.models import FastText, Word2Vec
from itertools import product
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
os.environ['PYTHONWARNINGS'] = 'ignore'


nltk.download('punkt')
nltk.download('punkt_tab')


C:\Users\HP Victus\AppData\Roaming\Python\Python313\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
[nltk_data] Downloading package punkt to C:\Users\HP
[nltk_data]     Victus\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\HP
[nltk_data]     Victus\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

### Splitting 

This section serves to split the dataset into Train, validation and test sets

In [2]:
numeric_cols = ["telecommuting", "missing_count", "total_text_len", "company_profile_len", "description_len", 
                "requirements_len", "benefits_len", "company_profile_word_count", "description_word_count", 
                "requirements_word_count", "benefits_word_count", "salary_provided", "has_company_profile",
                "vague_location", "has_company_logo", "has_questions"]

In [3]:
### loading clean text
df = pd.read_csv("./data/clean/fake_job_postings_ALL.csv")
df.head()


,full_text,telecommuting,missing_count,total_text_len,company_profile_len,description_len,requirements_len,benefits_len,company_profile_word_count,description_word_count,requirements_word_count,benefits_word_count,salary_provided,has_company_profile,vague_location,has_company_logo,has_questions,fraudulent
0,marketing intern us ny new york we are food 52...,0,3,2642,885,905,852,0,141,124,115,0,0,1,0,1,0,0
1,commissioning machinery assistant cma us ia we...,0,6,2597,879,355,1363,0,141,50,164,0,0,1,0,1,0,0
2,bill review manager us fl fort worth spot sour...,0,0,3926,1628,1520,757,21,207,168,89,3,0,1,0,1,1,0
3,head of content m f de be berlin founded in 20...,0,0,2538,881,433,764,460,133,57,77,65,1,1,0,1,1,0
4,hp bsm sme us fl pensacola solutions 3 is a wo...,0,3,1798,1364,75,359,0,192,5,45,0,0,1,0,1,1,0


In [4]:
# 1. Split 80% Train, 20% "Rest" (temp_data)
train_data, temp_data = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['fraudulent']
)

# 2. Split that 20% into half (10% Val, 10% Test)
# FIX: Use temp_data['fraudulent'] for stratification
val_data, test_data = train_test_split(
    temp_data, test_size=0.5, random_state=42, stratify=temp_data['fraudulent']
)

In [5]:
X_train_text = train_data['full_text']
X_train_numeric = train_data[numeric_cols]
y_train = train_data['fraudulent']

X_val_text = val_data['full_text']
X_val_numeric = val_data[numeric_cols]
y_val = val_data['fraudulent']

X_test_text = test_data['full_text']
X_test_numeric = test_data[numeric_cols]
y_test = test_data['fraudulent']

### Tokenization

This sections serves to tokenize free text into a sequence of integers

### Embeddings

This section serves to convert the token ids into a high-dimensional vector to capture the semantic and syntactic meaning of the tokens

We will try a pretrained CBow as well as using FastText from scratch and evaluate their performance. 

#### FastText (using skipgram)

In [6]:
sentences = [word_tokenize(text.lower()) for text in X_train_text]  

In [8]:
fasttext_model = FastText(vector_size=100, window=5, min_count=5, sg=1) #assuming we want a 100 word vector
fasttext_model.build_vocab(sentences)
fasttext_model.train(sentences, total_examples=fasttext_model.corpus_count, epochs=10)

(29316023, 38782700)

Some checks if we we manage to learn technical jargon and correlated words

In [7]:
fasttext_model_optimal = FastText(vector_size=100, window=3, min_count=2, epochs=10, negative=5, sg = 1) 
fasttext_model_optimal.build_vocab(sentences)
fasttext_model_optimal.train(sentences, total_examples=fasttext_model_optimal.corpus_count, epochs=10)

(29541780, 38782700)

In [8]:
## Save the trained optimal FastText models
fasttext_model_optimal.save("optimal_fasttext.bin")

In [ ]:
# Guide to how to extract the weights from the saved bin file.

# fasttext_model_optimal = FastText.load("optimal_fasttext.bin")

# # 2. Extract the weight matrix (NumPy array)
# # This matrix has shape [vocab_size + bucket_size, dim]
# weights = fasttext_model_optimal.get_input_matrix()

# # 3. Convert to PyTorch tensor
# weights_tensor = torch.from_numpy(weights)

# # 4. Create the nn.Embedding layer
# # Set freeze=False if you want to continue fine-tuning in PyTorch
# embedding_layer = nn.Embedding.from_pretrained(weights_tensor, freeze=True)

In [12]:
print(fasttext_model.wv['saas']) #technical jargon
print(fasttext_model.wv.vectors.shape) 

[-0.44127873  0.29269943 -0.20992798  0.0274798  -0.23092476 -0.32069984
  0.41804054 -0.18136458 -0.1894433  -0.1362076   0.00914257 -0.1966062
  0.20498233  0.67763853 -0.1728017   0.25251487  0.74282527  0.3525357
  0.17080547 -0.77085024 -0.75780195 -0.48803815 -0.3457629   0.24941416
  0.31104037 -0.58045834  0.05023273 -0.2795498  -0.32412434 -0.5422581
  0.07484528  0.3401077   0.34700844 -0.2753096   0.3083525   0.15955493
  0.1275885  -0.21626562 -0.4201954   0.26751935  0.48697776 -0.58222204
  0.7250583  -0.32693085  0.01717435  0.3978424  -0.08834979  0.2411034
  0.01113499  0.2888671   0.6228739  -0.35008714  0.43305796 -0.3201334
 -0.22850345  0.4945031   0.1694083  -0.20557703 -0.5008402   0.05236493
 -0.15081197  0.08727212  0.180482   -0.17808455 -0.45463505  0.7670393
  0.21150579 -0.10523701 -0.22565837  0.2534242  -0.26881006 -0.065399
  0.3988634  -0.07600085 -0.64498     0.33370078  0.22813998  0.3183136
  0.04969181 -0.21199797 -0.30767307 -0.4708828  -0.05592019

In [13]:
print(fasttext_model.wv['bingsu']) #checking for oov
print(fasttext_model.wv.vectors.shape) 

[-0.23686692  0.2818083  -0.40928096  0.21932498  0.01980837 -0.29707867
 -0.3271298   0.23569815  0.01006085 -0.09854509  0.12812121  0.27201435
 -0.38270295  0.4274572  -0.08210943 -0.13433227 -0.08935854 -0.16407348
 -0.06535226 -0.05687969 -0.08372589 -0.25539878  0.24452172  0.5328888
 -0.25796324 -0.3205903  -0.21345083 -0.06703421  0.04172352  0.11684978
  0.06497709  0.11305131  0.2756554   0.30529216  0.20174132  0.44263345
  0.23003122 -0.14365298  0.21109204 -0.05862045  0.06520065 -0.1818783
  0.07786035 -0.15485898 -0.36972535 -0.13474202 -0.16494924 -0.04016281
  0.15612072  0.11966276  0.21661782  0.08624003  0.12540875  0.09655247
  0.09594531  0.10366593 -0.09116521  0.07041204 -0.07044877  0.16439514
 -0.2944317  -0.24429576  0.1452024  -0.12817104 -0.025415    0.28857398
  0.34442943 -0.40393212  0.2622219   0.2788963  -0.22086093  0.08236961
  0.3978415  -0.21478513  0.341131   -0.07938153 -0.03152023 -0.34124857
  0.28266168  0.01746918 -0.5938349   0.04660112 -0.1

In [14]:
print(fasttext_model.wv.similarity('software', 'engineer'))
print(fasttext_model.wv.similarity('skills', 'experience'))
print(fasttext_model.wv.most_similar('water', topn=10))

print(fasttext_model_optimal.wv.similarity('software', 'engineer'))
print(fasttext_model_optimal.wv.similarity('skills', 'experience'))
print(fasttext_model_optimal.wv.most_similar('water', topn=10))


0.43667734
0.5359996
[('wastewater', 0.7438775897026062), ('waters', 0.7159039974212646), ('deepwater', 0.6504371166229248), ('carpet', 0.6377760171890259), ('waterloo', 0.6371378898620605), ('refrigeration', 0.6138995289802551), ('pumping', 0.6138473153114319), ('lube', 0.6129937171936035), ('dry', 0.60990309715271), ('watervliet', 0.6004721522331238)]
0.42343098
0.54381937
[('saltwater', 0.8272857069969177), ('wastewater', 0.8055526614189148), ('backwater', 0.7913426160812378), ('watering', 0.7908748388290405), ('groundwater', 0.7908555269241333), ('stormwater', 0.781193196773529), ('stillwater', 0.7718286514282227), ('whitewater', 0.7634021639823914), ('waters', 0.7453770041465759), ('deepwater', 0.7343776226043701)]


#### CBOW

In [ ]:

cbow_model = Word2Vec(vector_size=100, window=5, min_count=5, sg=0) #assuming we want a 100 word vector
cbow_model.build_vocab(sentences)
cbow_model.train(sentences, total_examples=cbow_model.corpus_count, epochs=10)

In [ ]:
print(cbow_model.wv.similarity('software', 'engineer'))
print(cbow_model.wv.similarity('skills', 'experience'))
cbow_model.wv.most_similar('research', topn=10)


### Numerical Features Variance Monitoring

We are going to compare the variance between the fake and real datasets so that we can find out which columns are not separable and thus not useful for the model to pick up signals from.

In [ ]:
def audit_numerical_features(dataloader, feature_names=None):
        import numpy as np

        all_features = []
        all_labels   = []

        for inputs, targets in dataloader:
            all_features.append(inputs['numerical_features'].numpy())
            all_labels.append(targets.numpy())

        X = np.vstack(all_features)   # (N, num_features)
        y = np.concatenate(all_labels)

        print(f"Shape: {X.shape}  |  Fake rate: {y.mean():.3f}\n")

        names = feature_names or [f"feat_{i}" for i in range(X.shape[1])]

        issues = []
        for i, name in enumerate(names):
            col      = X[:, i]
            variance = col.var()
            nan_pct  = np.isnan(col).mean() * 100
            scale    = np.abs(col).max()

            fake_mean = col[y == 1].mean() if (y == 1).any() else float('nan')
            real_mean = col[y == 0].mean() if (y == 0).any() else float('nan')
            sep       = abs(fake_mean - real_mean) / (col.std() + 1e-8)  # Cohen's d approx

            flag = []
            if variance < 1e-4:      flag.append("NEAR-ZERO VARIANCE")
            if nan_pct > 0:          flag.append(f"{nan_pct:.1f}% NaN")
            if scale > 100:          flag.append(f"LARGE SCALE (max={scale:.0f}) — needs normalisation")
            if sep < 0.1:            flag.append("LOW SEPARABILITY (Cohen's d < 0.1)")

            status = " | ".join(flag) if flag else "ok"
            print(f"  {name:<30} var={variance:8.4f}  sep={sep:.3f}  {status}")
            if flag:
                issues.append(name)

        print(f"\n{len(issues)}/{len(names)} features flagged")
        return issues

### Hyperparameter tuning

We are going to tune the `sliding window size`, `embedding vector size`,`epochs`, `negative sampling` to obtain the optimal custom embedding which would be easily plugged into our model

we will measure through extrinsic evaluation and intrinsic evaluation

In [ ]:
#to check oov rate
def oov_rate(model, corpus):
    oov = sum(1 for w in corpus if w not in model.wv)
    return oov/len(corpus)

#check top 10 words are similar to each other
def nearest_neighbour(model, test_words, topn = 10):
    scores = []
    for word in test_words:
        try:
            neighbours = model.wv.most_similar(word, topn=topn)
            scores.append(np.mean([score for _, score in neighbours]))
        except KeyError:
            pass
    return np.mean(scores)

#check if 2 correlated and 2 uncorrelated words are similar
def analogy_score(model, test_cases):
    correct = 0
    for pos1, pos2, neg1, expected in test_cases:
        try:
            results = model.wv.most_similar(
                positive=[pos1, pos2], negative=[neg1], topn=5
            )
            predicted = [w for w, _ in results]
            if expected in predicted:
                correct += 1
        except KeyError:
            pass
    return correct / len(test_cases)

In [ ]:
#DO NOT RUN THIS IT WILL TAKE 10.5 HOURS

param_grid = {
    'vector_size': [100, 200, 300],
    'window':      [3, 5, 10],
    'min_count':   [2, 5],
    'epochs':      [10, 20],
    'negative':    [5, 10],
}

test_words = ['engineer', 'manager', 'python', 'healthcare', 'experience', 'salary']
job_analogies = [
    ('engineer', 'python', 'manager', 'java'),
    ('senior', 'engineer', 'junior', 'developer'),
    ('full_time', 'salary', 'part_time', 'hourly'),
    ('healthcare', 'nurse', 'finance', 'analyst'),
]

results = []

keys = list(param_grid.keys())
combos = list(product(*param_grid.values()))
print(f"Total combinations: {len(combos)} x 2 models = {len(combos)*2} runs")

for combo in combos:
    params = dict(zip(keys, combo))

    for model_type in ['fasttext', 'cbow']:
        if model_type == 'fasttext':
            model = FastText(**params, sg=1, min_n=3, max_n=6)
        else:
            model = Word2Vec(**params, sg=0)

        model.build_vocab(sentences)
        model.train(sentences, total_examples=model.corpus_count, epochs=params['epochs'])

        coherence = nearest_neighbour(model, test_words)
        oov       = oov_rate(model, vocab)
        analogy   = analogy_score(model, job_analogies)

        results.append({
            **params,
            'model_type': model_type,
            'coherence':  round(coherence, 4),
            'oov_rate':   round(oov, 4),
            'analogy':    round(analogy, 4),
        })
        print(f"[{model_type}] {params} → coherence={coherence:.4f}, oov={oov:.4f}, analogy={analogy:.4f}")

results_df = pd.DataFrame(results)
results_df.to_csv('data/clean/embedding_tuning.csv', index=False)